<a href="https://colab.research.google.com/github/stfnnnnnnn/karl-mangahas-flyrank/blob/main/work/notebooks/w06_validation_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/stfnnnnnnn/1st-act/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*



### Finding A — Finding #1: The Anatomy of Growing Content

>The paper compares pages with rising impressions against pages with falling impressions. It reports that growing content averaged **3,180 words versus 2,311 words** for declining content (37.6% longer) and **184 days versus 230 days** in age (20% younger). The comparison is large — about 74K rising pages versus 45K falling pages — and the paper appropriately describes the result as an **observational comparison**. Trend direction is defined elsewhere in the paper from the most recent 30-day impression change versus the previous 30 days.

**Methodology question: where does the label come from, and what is independent of it?** *How exactly was the growing-versus-declining label constructed for each page, and are all variables being compared measured independently of the period used to assign that label?*

>Because the label comes from recent 30-day versus prior-30-day impression movement, I would want the analysis to make the timeline explicit. I would also ask whether the age and word-count gaps remain within the same brand/client, or under a client-grouped analysis. That would help distinguish a portfolio-level association from differences in client mix, publishing strategy, or content population.

**Safe interpretation:** In this portfolio, growing and declining pages **showed measured differences** in age and depth. The result is useful as a directional signal for investigation, but it does not show that making a page longer or younger would cause growth.

### Finding B — Finding #4: The Freshness Multiplier

>The paper identifies **31–90 days since update** as the strongest stable freshness band, with a reported growth-to-decline ratio of **7.88:1**. It also reports that 365+ day content refreshed within 30 days showed a **3.2× health-score difference** (10.7 to 34.5) and roughly **57× higher impressions** (71 to 4,039). Importantly, the paper also warns against over-reading the 361+ freshness bucket because its 283:1 ratio is based on only one declining page.

**Methodology question — does the design support an intervention claim?** *Were mature pages that received a recent refresh comparable to mature pages that did not before the refresh occurred?*

>Refresh assignment is unlikely to be random: teams may preferentially refresh pages with stronger historical demand, strategic importance, or better prior visibility. I would therefore ask whether the comparison controls for pre-refresh performance, client, age, and prior visibility, and whether the outcome measurements occur strictly after the refresh. A matched within-client or time-aware pre/post design with comparable unrefreshed pages would better separate the refresh association from selection effects.

**Safe interpretation:**

>The paper **observed a large cohort difference** between recently refreshed and less-recently updated mature pages. That supports refresh status as a decision-support signal for review prioritization; by itself, the observational comparison does not establish that the refresh caused the measured lift.

### Why these are useful methodology questions

>The first asks **where the outcome label comes from and whether the comparison respects that construction**. The second asks **whether the validation/comparison design can carry an intervention-style interpretation**. Both follow the same standard I apply to my Week-5 model below: make the timeline explicit, test repeated entities honestly, inspect leakage, and keep the wording no stronger than the evidence.



In [1]:
# Reference values transcribed from the supplied March 2026 FlyRank paper.
# These constants make the markdown summary above easy to audit; they are not model inputs.
paper_checks = {
    "finding_1_growing_avg_words": 3180,
    "finding_1_declining_avg_words": 2311,
    "finding_1_growing_avg_age_days": 184,
    "finding_1_declining_avg_age_days": 230,
    "finding_4_stable_freshness_growth_to_decline_ratio": 7.88,
    "finding_4_mature_refresh_health_before": 10.7,
    "finding_4_mature_refresh_health_after": 34.5,
    "finding_4_mature_refresh_health_difference_x": 3.2,
    "finding_4_mature_refresh_impressions_comparison": "71 vs 4039 (~57x)",
}
paper_checks

{'finding_1_growing_avg_words': 3180,
 'finding_1_declining_avg_words': 2311,
 'finding_1_growing_avg_age_days': 184,
 'finding_1_declining_avg_age_days': 230,
 'finding_4_stable_freshness_growth_to_decline_ratio': 7.88,
 'finding_4_mature_refresh_health_before': 10.7,
 'finding_4_mature_refresh_health_after': 34.5,
 'finding_4_mature_refresh_health_difference_x': 3.2,
 'finding_4_mature_refresh_impressions_comparison': '71 vs 4039 (~57x)'}

## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

### What changed from Week 5

>In Week 5 I already used `GroupShuffleSplit` by `client_hash_id`, which was a good step because it tested the model on clients it had not seen during training. While reviewing the feature timeline for this audit, though, I found a separate problem that matters just as much as the split.

My Week-5 feature set included `imp_prev30`, `clk_prev30`, and **`pos_last30`**. The decline label compares impressions in the last 30 days with impressions in the previous 30 days. That means `pos_last30` was measured during the same period I was trying to predict. It is not the label itself, but it gives the model information from the outcome window.

I corrected that by rebuilding average position as **`pos_prev30`**. I then ran the same Random Forest two ways so I could see what the validation choice changes:

1. **Before — random row split:** pages from the same client can appear on both sides.
2. **After — grouped client split:** whole clients are held out.

> On the March development slice I ended up with **82,025 pages across 37 clients**, with an overall decline rate of **26.87%**. Under the random split, ROC-AUC was **0.557** and Average Precision was **0.307**. Under the client-grouped split, ROC-AUC fell to **0.532** and Average Precision to **0.283**. Precision@20 changed from **45% to 40%**, while Precision@50 changed from **36% to 40%**.

>The part I find most important is the client overlap: the random split had **34 clients represented on both sides**, while the grouped split had **zero** overlap. The drop in ROC-AUC is small but meaningful. It tells me that some of the apparent performance from a random row split does not transfer as well when I ask the harder question: *can this model rank pages for clients it has never seen?*

>I will therefore use the **grouped result** for my claims. I am keeping the random result only as a comparison, not as the headline performance.


In [2]:
%pip -q install duckdb huggingface_hub

import os, getpass, duckdb, numpy as np, pandas as pd
from IPython.display import display

HF_TOKEN = os.environ.get("HF_TOKEN")
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass("HF token: ")

con = duckdb.connect()
con.execute(f"""
CREATE OR REPLACE SECRET hf (
    TYPE huggingface,
    TOKEN '{HF_TOKEN}'
)
""")

REL = "hf://datasets/FlyRank/internship-warehouse"
FACT = f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')"

# Same March development endpoint used in Week 5, but with position rebuilt in the prior window.
features = con.sql(f"""
WITH bounds AS (
    SELECT MAX(report_date) AS end_d
    FROM {FACT}
    WHERE month = '2026-03'
),
windowed AS (
    SELECT
        f.client_hash_id,
        f.content_hash_id,
        SUM(CASE WHEN f.report_date >  b.end_d - INTERVAL 30 DAY
                 THEN f.gsc_impressions ELSE 0 END) AS imp_last30,
        SUM(CASE WHEN f.report_date <= b.end_d - INTERVAL 30 DAY
                 THEN f.gsc_impressions ELSE 0 END) AS imp_prev30,
        SUM(CASE WHEN f.report_date <= b.end_d - INTERVAL 30 DAY
                 THEN f.gsc_clicks ELSE 0 END) AS clk_prev30,
        AVG(CASE WHEN f.report_date <= b.end_d - INTERVAL 30 DAY
                 THEN f.gsc_avg_position END) AS pos_prev30,
        AVG(CASE WHEN f.report_date >  b.end_d - INTERVAL 30 DAY
                 THEN f.gsc_avg_position END) AS pos_last30
    FROM {FACT} f, bounds b
    WHERE f.report_date > b.end_d - INTERVAL 60 DAY
      AND f.report_date <= b.end_d
    GROUP BY f.client_hash_id, f.content_hash_id
    HAVING imp_prev30 >= 100
)
SELECT * FROM windowed
""").df()

features["is_declining"] = (
    features["imp_last30"] < 0.8 * features["imp_prev30"]
).astype(int)

print(f"Rows: {len(features):,}")
print(f"Clients: {features['client_hash_id'].nunique():,}")
print(f"Overall decline base rate: {features['is_declining'].mean():.2%}")
features.head()


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows: 82,025
Clients: 37
Overall decline base rate: 26.87%


,client_hash_id,content_hash_id,imp_last30,imp_prev30,clk_prev30,pos_prev30,pos_last30,is_declining
0,client_73cda7b4e4f265ea,content_c04dbfea57e94e79,144.0,166.0,1.0,4.485106,4.057307,0
1,client_73cda7b4e4f265ea,content_ea3acd89aac1237f,7721.0,8430.0,8.0,5.045716,5.239828,0
2,client_73cda7b4e4f265ea,content_2b8cc8f206dbecb3,297.0,357.0,0.0,7.407040,10.931267,0
3,client_73cda7b4e4f265ea,content_adbaad407e1d729c,4487.0,7783.0,35.0,1.704329,2.250983,1
4,client_73cda7b4e4f265ea,content_1ff6464078028aa5,219.0,173.0,1.0,8.770291,8.511006,0


In [3]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.metrics import roc_auc_score, average_precision_score

SAFE_FEATURES = ["imp_prev30", "clk_prev30", "pos_prev30"]
X = features[SAFE_FEATURES].copy()
X["pos_prev30"] = X["pos_prev30"].fillna(0)
y = features["is_declining"].copy()
groups = features["client_hash_id"]

def precision_at_k(y_true, scores, k):
    k = min(k, len(scores))
    order = np.argsort(-np.asarray(scores))[:k]
    return np.asarray(y_true)[order].mean()

def fit_eval(train_idx, test_idx, label):
    model = RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1)
    model.fit(X.iloc[train_idx], y.iloc[train_idx])
    prob = model.predict_proba(X.iloc[test_idx])[:, 1]
    yt = y.iloc[test_idx]
    return model, prob, {
        "Validation": label,
        "Test rows": len(test_idx),
        "Test clients": groups.iloc[test_idx].nunique(),
        "Base rate": yt.mean(),
        "ROC-AUC": roc_auc_score(yt, prob),
        "Average Precision": average_precision_score(yt, prob),
        "Precision@20": precision_at_k(yt, prob, 20),
        "Precision@50": precision_at_k(yt, prob, 50),
    }

# BEFORE: ordinary row-random split.
idx = np.arange(len(features))
tr_r, te_r = train_test_split(idx, test_size=0.25, random_state=42, stratify=y)
rf_random, prob_random, before = fit_eval(tr_r, te_r, "Before: random row split")

# AFTER: whole clients held out.
gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
tr_g, te_g = next(gss.split(X, y, groups))
rf_grouped, prob_grouped, after = fit_eval(tr_g, te_g, "After: grouped client split")

split_comparison = pd.DataFrame([before, after])
display(split_comparison.style.format({
    "Base rate": "{:.2%}", "ROC-AUC": "{:.3f}", "Average Precision": "{:.3f}",
    "Precision@20": "{:.2%}", "Precision@50": "{:.2%}"
}))

print("Train/test client overlap — random:", len(set(groups.iloc[tr_r]) & set(groups.iloc[te_r])))
print("Train/test client overlap — grouped:", len(set(groups.iloc[tr_g]) & set(groups.iloc[te_g])))


,Validation,Test rows,Test clients,Base rate,ROC-AUC,Average Precision,Precision@20,Precision@50
0,Before: random row split,20507,34,26.87%,0.557,0.307,45.00%,36.00%
1,After: grouped client split,26184,10,25.96%,0.532,0.283,40.00%,40.00%


Train/test client overlap — random: 34
Train/test client overlap — grouped: 0


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

### Timeline

`previous 30 days (features)  →  decision point  →  most recent 30 days (label/outcome)`

### Final feature audit

| Field | Role | Prediction-time status | Decision |
|---|---|---|---|
| `client_hash_id` | grouping key | known | split/group only; never a feature |
| `content_hash_id` | page key | known | joins/error inspection only; never a feature |
| `imp_prev30` | feature | prior window | keep |
| `clk_prev30` | feature | prior window | keep |
| `pos_prev30` | feature | prior window | keep |
| `imp_last30` | label ingredient | outcome window | exclude from X |
| `pos_last30` | Week-5 feature | **outcome window** | remove from final X |
| `is_declining` | target | derived from last vs previous impressions | target only |

>The main issue I found was `pos_last30`. I would not call it direct label leakage because the label is based on impressions, not position. The problem is **time leakage**: I was allowing the model to see search position from the same 30-day period in which the decline happened.

>That distinction matters to me because the Week-5 feature did not look obviously wrong just from its name. It only became a problem once I placed it on the feature/target timeline.

>I also checked for the other leakage risks from the course guidance. I am not using FlyRank product scores or optimization flags as model inputs, and neither client nor content IDs are model features. My final model uses only `imp_prev30`, `clk_prev30`, and `pos_prev30`.

>The next code cell deliberately compares the overlapping Week-5-style feature set with this audited feature set. I am treating that comparison as an **attack test**, not as another model-selection exercise.


In [4]:
# Attack test: quantify how much the overlapping Week-5 position feature changes grouped performance.
from sklearn.metrics import roc_auc_score, average_precision_score

X_leaky = features[["imp_prev30", "clk_prev30", "pos_last30"]].copy()
X_leaky["pos_last30"] = X_leaky["pos_last30"].fillna(0)

rf_leaky = RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1)
rf_leaky.fit(X_leaky.iloc[tr_g], y.iloc[tr_g])
prob_leaky = rf_leaky.predict_proba(X_leaky.iloc[te_g])[:, 1]

yt = y.iloc[te_g]
leakage_comparison = pd.DataFrame([
    {
        "Feature set": "Week-5 style: includes pos_last30 (overlap)",
        "ROC-AUC": roc_auc_score(yt, prob_leaky),
        "Average Precision": average_precision_score(yt, prob_leaky),
        "Precision@20": precision_at_k(yt, prob_leaky, 20),
        "Precision@50": precision_at_k(yt, prob_leaky, 50),
    },
    {
        "Feature set": "Audited: pos_prev30 only",
        "ROC-AUC": roc_auc_score(yt, prob_grouped),
        "Average Precision": average_precision_score(yt, prob_grouped),
        "Precision@20": precision_at_k(yt, prob_grouped, 20),
        "Precision@50": precision_at_k(yt, prob_grouped, 50),
    },
])
print(f"Grouped-test base rate: {yt.mean():.2%}")
display(leakage_comparison.style.format({
    "ROC-AUC": "{:.3f}", "Average Precision": "{:.3f}",
    "Precision@20": "{:.2%}", "Precision@50": "{:.2%}"
}))

# Feature importance sanity check on the audited model.
importance = pd.Series(rf_grouped.feature_importances_, index=SAFE_FEATURES).sort_values(ascending=False)
display(importance.to_frame("Importance"))


Grouped-test base rate: 25.96%


,Feature set,ROC-AUC,Average Precision,Precision@20,Precision@50
0,Week-5 style: includes pos_last30 (overlap),0.530,0.295,95.00%,98.00%
1,Audited: pos_prev30 only,0.532,0.283,40.00%,40.00%


,Importance
pos_prev30,0.537396
imp_prev30,0.408429
clk_prev30,0.054175


### Real failure examples

>The aggregate metrics tell me whether the ranking has signal, but they do not tell me what the model gets wrong. I therefore inspect false positives and false negatives from the **audited grouped model**.

>A **false positive** is a page the model ranks as likely to decline even though it does not meet my decline definition in the outcome window. In practice, that could waste a reviewer’s limited time.

>A **false negative** is more concerning for this use case: the page actually declines under my definition, but the model gives it a low enough score that it may not be reviewed.

>I am deliberately keeping these examples pseudonymized. I also do not try to invent a story for why an individual page failed. With only prior impressions, clicks, and position, there are many things the model cannot see — seasonality, content changes, competing pages, query mix, or client-specific events, for example.

>What I want from this section is simpler: **do the mistakes reveal a pattern that tells me where this three-feature model is too limited?** The counts and examples below are evidence for that discussion, not proof of the cause of any individual error.



In [5]:
# Error examples from the audited grouped model.
err = features.iloc[te_g][[
    "client_hash_id", "content_hash_id", "imp_prev30", "clk_prev30", "pos_prev30", "is_declining"
]].copy()
err["prob_decline"] = prob_grouped
err["predicted"] = (err["prob_decline"] >= 0.5).astype(int)

false_pos = err[(err["is_declining"] == 0) & (err["predicted"] == 1)].sort_values("prob_decline", ascending=False)
false_neg = err[(err["is_declining"] == 1) & (err["predicted"] == 0)].sort_values("prob_decline", ascending=True)

print("False positives:", len(false_pos))
display(false_pos.head(5))
print("False negatives:", len(false_neg))
display(false_neg.head(5))

# Compact error-rate summary.
print("Audited grouped test rows:", len(err))
print("False-positive rate among actual negatives:",
      f"{len(false_pos) / max(1, (err['is_declining'] == 0).sum()):.2%}")
print("False-negative rate among actual positives:",
      f"{len(false_neg) / max(1, (err['is_declining'] == 1).sum()):.2%}")


False positives: 2851


,client_hash_id,content_hash_id,imp_prev30,clk_prev30,pos_prev30,is_declining,prob_decline,predicted
19048,client_fef1a8f436438636,content_0fc93224cf2ba5f0,165.0,0.0,47.680490,0,0.930000,1
72784,client_62f4a7e64f5e0096,content_4b268f35547aab0d,5596.0,2.0,2.544062,0,0.923333,1
53621,client_62f4a7e64f5e0096,content_ed1b332b1cf85f2b,1256.0,0.0,0.294339,0,0.923333,1
53356,client_62f4a7e64f5e0096,content_d8beae7f090b2c40,161.0,0.0,3.254196,0,0.920000,1
53095,client_62f4a7e64f5e0096,content_17ca9e02e3a45fe0,2144.0,0.0,3.700079,0,0.920000,1


False negatives: 5603


,client_hash_id,content_hash_id,imp_prev30,clk_prev30,pos_prev30,is_declining,prob_decline,predicted
54140,client_62f4a7e64f5e0096,content_1fe8a8a4e913881f,178.0,0.0,5.306667,1,0.000000,0
54494,client_62f4a7e64f5e0096,content_0fcdf0deb75b03d2,100.0,0.0,7.522928,1,0.000000,0
13934,client_62f4a7e64f5e0096,content_499ff4ad3829b26f,232.0,1.0,5.884966,1,0.000000,0
81766,client_a80fca3f171ed1de,content_6a4ada91f5b0603a,134.0,0.0,15.481954,1,0.003333,0
71514,client_62f4a7e64f5e0096,content_dca89ae01130f444,141.0,0.0,8.715632,1,0.003333,0


Audited grouped test rows: 26184
False-positive rate among actual negatives: 14.71%
False-negative rate among actual positives: 82.42%


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

### What I said in Week 5

> “Both Logistic Regression and Random Forest achieved perfect Precision@20 and Precision@50 on the grouped validation split.”

The number itself came from the run, but after this audit I would not present it as evidence that the model was performing extremely well. The feature set still contained `pos_last30`, which came from the outcome window. Also, seeing perfect top-K precision beside a ROC-AUC close to random should have made me investigate the result before treating it as a success.

### How I would say it now

>In the March development slice, my Week-5 model produced very high Precision@20 and Precision@50 on a client-grouped holdout. During this audit, I found that one of its inputs — average position from the last 30 days — overlapped the period used to define decline. I rebuilt that feature from the previous 30-day window and reran the model.

>With the leakage-safe features, the client-grouped Random Forest measured **0.532 ROC-AUC and 0.283 Average Precision**, with **40% Precision@20 and 40% Precision@50** on a test set whose decline base rate was **25.96%**. These results are much more modest than the Week-5 top-K numbers, but they are also more useful because the test better matches the decision I actually want to make.

>What I can say is that the audited model shows **some directional ranking signal above the test-set base rate at the top of the queue**, but overall discrimination is weak. I would use it as a **decision-support ranking for human review**, not as an automatic prediction that a page will decline.

>I also cannot claim that acting on a flagged page will improve its performance. This notebook measures whether historical search signals help prioritize future decline risk; it does not test the causal effect of refreshing or editing a page.

### What changed in my interpretation

- **Observed:** the final features come only from the period before the decline window.
- **Measured:** the grouped holdout has no client overlap, and I report its base rate beside the model metrics.
- **Directional:** Precision@20 and Precision@50 are above the grouped test base rate, but ROC-AUC is only slightly above 0.5, so I would not describe the model as broadly strong.
- **Decision-support:** the model can help order a review queue, but the final decision still needs a person who can inspect the page and context.

The biggest lesson from this audit is that a weaker but believable number is more valuable than a perfect-looking number produced by a questionable feature timeline.


In [6]:
# Final automated audit receipts.
assert "pos_last30" not in SAFE_FEATURES
assert "imp_last30" not in SAFE_FEATURES
assert "is_declining" not in SAFE_FEATURES
assert "client_hash_id" not in SAFE_FEATURES
assert "content_hash_id" not in SAFE_FEATURES
assert len(set(groups.iloc[tr_g]) & set(groups.iloc[te_g])) == 0

print("PASS: target-window fields excluded from final features")
print("PASS: IDs excluded from model features")
print("PASS: grouped train/test client overlap = 0")
print("PASS: final feature set =", SAFE_FEATURES)

PASS: target-window fields excluded from final features
PASS: IDs excluded from model features
PASS: grouped train/test client overlap = 0
PASS: final feature set = ['imp_prev30', 'clk_prev30', 'pos_prev30']


## Self-check

Before you submit, confirm each line honestly:

- [ - ] Every section above is filled — markdown thinking AND the code that backs it
- [ - ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ - ] No client names, URLs, or private queries anywhere
- [ - ] My claims use careful words: observed, measured, directional, decision-support
- [ - ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.